In [1]:
import pandas as pd
df = pd.read_csv('fake_job_postings.csv')
print(df.shape)
print(df.columns.tolist())
print(df['fraudulent'].value_counts())

(17880, 18)
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
fraudulent
0    17014
1      866
Name: count, dtype: int64


In [2]:
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    text = re.sub('&amp;', ' and ', text)
    text = re.sub('<.*?>', ' ', text)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_description'] = df['description'].apply(clean_text)
df['clean_requirements'] = df['requirements'].apply(clean_text)
df['clean_company_profile'] = df['company_profile'].apply(clean_text)

df['full_text'] = df['clean_description'] + ' ' + df['clean_requirements'] + ' ' + df['clean_company_profile']
print(df['full_text'].iloc[0][:500])

food a fast growing james beard award winning online food community and crowd sourced and curated recipe hub is currently interviewing full and part time unpaid interns to work in a small team of editors executives and developers in its new york city headquarters reproducing and or repackaging existing food content for a number of partner sites such as huffington post yahoo buzzfeed and more in their various content management systems researching blogs and websites for the provisions by food aff


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['full_text'])

print(X.shape)
print(vectorizer.get_feature_names_out()[:20])

(17880, 5000)
['aa' 'aabbf' 'aan' 'ab' 'abc' 'abilities' 'ability' 'able' 'abreast'
 'abroad' 'absolute' 'absolutely' 'ac' 'academic' 'academy' 'acc'
 'accelerate' 'accelerator' 'accept' 'acceptable']


In [4]:
from sklearn.model_selection import train_test_split

y = df['fraudulent']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train size:", X_train.shape, "Test size:", X_test.shape)

Train size: (14304, 5000) Test size: (3576, 5000)


In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("Original fraud count:", sum(y_train))
print("After SMOTE fraud count:", sum(y_train_balanced))

Original fraud count: 693
After SMOTE fraud count: 13611


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf_model_smote = RandomForestClassifier(n_estimators=200, random_state=42)
rf_model_smote.fit(X_train_balanced, y_train_balanced)

print("Training complete!")

Training complete!


In [7]:
from sklearn.metrics import classification_report

rf_smote_probs = rf_model_smote.predict_proba(X_test)[:, 1]
rf_smote_pred_04 = (rf_smote_probs >= 0.4).astype(int)

print("FINAL RESULTS (SMOTE + Threshold 0.4):")
print(classification_report(y_test, rf_smote_pred_04))

FINAL RESULTS (SMOTE + Threshold 0.4):
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3403
           1       0.86      0.71      0.78       173

    accuracy                           0.98      3576
   macro avg       0.92      0.85      0.88      3576
weighted avg       0.98      0.98      0.98      3576



In [8]:
import joblib

joblib.dump(rf_model_smote, 'fraud_detector_model_final.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer_final.pkl')

print("Model saved! Approach: Random Forest + SMOTE, Threshold: 0.4")

Model saved! Approach: Random Forest + SMOTE, Threshold: 0.4
